# 01 — Bronze: descarga

**Fase 1 · Capa Bronze** · interpretación en `data/bronze/bronze.md`

Lee `config.yaml → fuentes` y descarga cada fuente en `data/bronze/<fuente>/<AAAA-MM-DD>/`. Luego registra cada archivo en `data/bronze/manifest.json` con su fuente, URL, fecha, tamaño, sha256 y n° de filas.

| Fuente | Qué se descarga |
| --- | --- |
| F1 SIAF | 2009–2024 anual y 2025–2026 *Diario* (el *Mensual* se descarta) |
| F2 SISMEPRE | 7 CSV (se publican completos, no por año) |
| F3 RENAMU | ediciones 2009–2026; los ZIP se guardan y se descomprimen en la misma carpeta (el análisis usa 2020–2026) |


## 1. Configuración

In [1]:
import sys
from pathlib import Path

RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ARQUITECTURA.md").exists())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
    
import json, shutil, uuid
from datetime import datetime
from zoneinfo import ZoneInfo

import pandas as pd
from tqdm.auto import tqdm

from src.config import cargar_config, cargar_fuentes, cargar_reglas, ruta
from src import bronze

import requests

cfg = cargar_config()
FECHA = datetime.now(ZoneInfo(cfg["spark"]["timezone"])).strftime("%Y-%m-%d")   # carpeta de la descarga
RUN_ID = f"bronze_descarga_{datetime.now(ZoneInfo('UTC')).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:6]}"
FORZAR = False        # True: descargar aunque el servidor no haya cambiado el archivo

print("Fecha de descarga:", FECHA)
print("Id de ejecución  :", RUN_ID)

Fecha de descarga: 2026-11-20
Id de ejecución  : bronze_descarga_20261121T030045Z_f2c118


/home/rodri/proyectos/Final_BI/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Catálogo de recursos

In [2]:
cargar_config.cache_clear()
recursos = bronze.construir_catalogo(cargar_fuentes())

df_catalogo = pd.DataFrame([{
    "fuente": r.fuente, "anio": r.anio, "archivo": r.archivo, "modo": r.modo,
    "descomprimir": r.descomprimir,
    "destino": str(r.destino(FECHA).relative_to(RAIZ)) if r.modo == "automatica" else f"data/bronze/{r.carpeta_bronze}/<fecha>/",
    "url": r.url,
} for r in recursos])

print(df_catalogo.groupby("fuente").size().rename("recursos").to_string())
df_catalogo

fuente
F1    18
F2     7
F3    19


,fuente,anio,archivo,modo,descomprimir,destino,url
0,F1,2009.0,2009-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2026-11-20/2009-Ingre...,https://fs.datosabiertos.mef.gob.pe/datastoref...
1,F1,2010.0,2010-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2026-11-20/2010-Ingre...,https://fs.datosabiertos.mef.gob.pe/datastoref...
2,F1,2011.0,2011-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2026-11-20/2011-Ingre...,https://fs.datosabiertos.mef.gob.pe/datastoref...
3,F1,2012.0,2012-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2026-11-20/2012-Ingre...,https://fs.datosabiertos.mef.gob.pe/datastoref...
4,F1,2013.0,2013-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2026-11-20/2013-Ingre...,https://fs.datosabiertos.mef.gob.pe/datastoref...
5,F1,2014.0,2014-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2026-11-20/2014-Ingre...,https://fs.datosabiertos.mef.gob.pe/datastoref...
6,F1,2015.0,2015-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2026-11-20/2015-Ingre...,https://fs.datosabiertos.mef.gob.pe/datastoref...
7,F1,2016.0,2016-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2026-11-20/2016-Ingre...,https://fs.datosabiertos.mef.gob.pe/datastoref...
8,F1,2017.0,2017-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2026-11-20/2017-Ingre...,https://fs.datosabiertos.mef.gob.pe/datastoref...
9,F1,2018.0,2018-Ingreso.csv,automatica,False,data/bronze/siaf_ingreso/2026-11-20/2018-Ingre...,https://fs.datosabiertos.mef.gob.pe/datastoref...


## 3. Descarga

El manifest se guarda después de cada recurso.

In [3]:
manifest = bronze.leer_manifest()
resultados = []
http = requests.Session()

for r in tqdm(recursos, desc="Descargando"):
    try:
        res = bronze.procesar_recurso(r, FECHA, manifest, http, forzar=FORZAR)
    except Exception as e:
        res = bronze.Resultado(r, "error", str(e))
    if res.entradas:
        for e in res.entradas:
            e["run_id"] = RUN_ID
        manifest["archivos"].extend(res.entradas)
        bronze.guardar_manifest(manifest)
    resultados.append(res)
    tqdm.write(f"{res.estado:<19} {r.fuente} {r.archivo:<40} {res.detalle}")

df_res = pd.DataFrame([{
    "fuente": x.recurso.fuente, "archivo": x.recurso.archivo, "estado": x.estado,
    "detalle": x.detalle, "archivos_registrados": len(x.entradas), "segundos": round(x.segundos, 1),
} for x in resultados])
print()
print(df_res["estado"].value_counts().to_string())

Descargando: 100%|████████████████████████████████████████████████████████████████████| 44/44 [00:00<00:00, 422.87it/s]

ya_existe           F1 2009-Ingreso.csv                         data/bronze/siaf_ingreso/2026-11-20/2009-Ingreso.csv
ya_existe           F1 2010-Ingreso.csv                         data/bronze/siaf_ingreso/2026-11-20/2010-Ingreso.csv
ya_existe           F1 2011-Ingreso.csv                         data/bronze/siaf_ingreso/2026-11-20/2011-Ingreso.csv
ya_existe           F1 2012-Ingreso.csv                         data/bronze/siaf_ingreso/2026-11-20/2012-Ingreso.csv
ya_existe           F1 2013-Ingreso.csv                         data/bronze/siaf_ingreso/2026-11-20/2013-Ingreso.csv
ya_existe           F1 2014-Ingreso.csv                         data/bronze/siaf_ingreso/2026-11-20/2014-Ingreso.csv
ya_existe           F1 2015-Ingreso.csv                         data/bronze/siaf_ingreso/2026-11-20/2015-Ingreso.csv
ya_existe           F1 2016-Ingreso.csv                         data/bronze/siaf_ingreso/2026-11-20/2016-Ingreso.csv
ya_existe           F1 2017-Ingreso.csv                         

## 4. Resumen de Bronze (desde el manifest)

In [4]:
manifest = bronze.leer_manifest()
df_m = pd.DataFrame(manifest["archivos"])
df_m["ext"] = df_m["archivo"].str.rsplit(".", n=1).str[-1].str.lower()

resumen = (df_m.groupby(["fuente", "origen", "ext"])
             .agg(archivos=("ruta", "count"), MB=("bytes", lambda s: round(s.sum() / 1e6, 1)),
                  filas=("filas", lambda s: int(s.fillna(0).sum())))
             .reset_index())
display(resumen)
print(f"Total Bronze: {len(df_m)} archivos · {df_m['bytes'].sum() / 1e9:.2f} GB")

,fuente,origen,ext,archivos,MB,filas
0,F1,descarga,csv,18,7335.4,15202372
1,F2,descarga,csv,7,59.1,493789
2,F3,descarga,zip,19,64.1,0
3,F3,extraido_de_zip,csv,1028,271.6,2240471
4,F3,extraido_de_zip,pdf,62,18.9,0


Total Bronze: 1134 archivos · 7.75 GB


In [5]:
# Detalle de SIAF y SISMEPRE (un archivo = una tabla)
display(df_m[df_m["fuente"].isin(["F1", "F2"])][["fuente", "anio", "archivo", "bytes", "filas", "last_modified", "fecha_carpeta"]]
        .assign(MB=lambda d: (d["bytes"] / 1e6).round(1)).drop(columns="bytes"))

,fuente,anio,archivo,filas,last_modified,fecha_carpeta,MB
1109,F1,2009.0,2009-Ingreso.csv,1235428.0,"Wed, 21 Oct 2026 03:52:23 GMT",2026-11-20,578.9
1110,F1,2010.0,2010-Ingreso.csv,1320104.0,"Wed, 21 Oct 2026 03:54:18 GMT",2026-11-20,622.4
1111,F1,2011.0,2011-Ingreso.csv,1348362.0,"Wed, 21 Oct 2026 03:57:12 GMT",2026-11-20,637.2
1112,F1,2012.0,2012-Ingreso.csv,759578.0,"Wed, 21 Oct 2026 04:23:47 GMT",2026-11-20,367.0
1113,F1,2013.0,2013-Ingreso.csv,721663.0,"Wed, 21 Oct 2026 20:37:07 GMT",2026-11-20,348.6
1114,F1,2014.0,2014-Ingreso.csv,753998.0,"Wed, 21 Oct 2026 21:21:46 GMT",2026-11-20,364.8
1115,F1,2015.0,2015-Ingreso.csv,781007.0,"Wed, 21 Oct 2026 22:51:01 GMT",2026-11-20,380.7
1116,F1,2016.0,2016-Ingreso.csv,764731.0,"Wed, 21 Oct 2026 23:52:34 GMT",2026-11-20,372.3
1117,F1,2017.0,2017-Ingreso.csv,683816.0,"Thu, 22 Oct 2026 01:34:24 GMT",2026-11-20,332.7
1118,F1,2018.0,2018-Ingreso.csv,656859.0,"Thu, 22 Oct 2026 02:23:29 GMT",2026-11-20,319.1


In [6]:
# RENAMU: CSV extraídos por edición
ren = df_m[(df_m["fuente"] == "F3") & (df_m["origen"] == "extraido_de_zip")]
display(ren.groupby("anio").agg(csv=("ext", lambda s: (s == "csv").sum()), pdf=("ext", lambda s: (s == "pdf").sum()),
                                MB=("bytes", lambda s: round(s.sum() / 1e6, 1)), filas_csv=("filas", lambda s: int(s.fillna(0).sum()))))

,csv,pdf,MB,filas_csv
anio,,,,
2009.0,98,3,18.1,211773
2010.0,98,3,17.0,205443
2011.0,98,3,17.6,205180
2012.0,108,4,21.2,220566
2013.0,107,4,20.9,218723
2014.0,96,3,18.0,196541
2015.0,115,6,19.8,213860
2016.0,111,6,18.3,207037
2017.0,116,6,21.9,279535


## 5. Reglas de calidad de Bronze

Se evalúan las reglas `BRZ-TODAS-*` de `config.yaml → reglas_calidad`.

In [7]:
FIRMAS_HTML = bronze.FIRMAS_HTML
reglas = {r["id"]: r for r in cargar_reglas()}
auto = [r for r in recursos if r.modo == "automatica"]
urls_ok = {e["url"] for e in manifest["archivos"] if e["origen"] == "descarga" and (RAIZ / e["ruta"]).exists()}

def _es_html(p):
    with open(p, "rb") as f:
        return f.read(512).lstrip().lower().startswith(FIRMAS_HTML)

evaluaciones = []
def registrar(rid, evaluadas, fallidas, detalle=""):
    r = reglas[rid]
    estado = "OK" if not fallidas else ("FALLA" if r["severidad"] == "critica" else "ALERTA")
    evaluaciones.append({"run_id": RUN_ID, "fecha_utc": bronze._ahora(), "capa": "bronze", "regla": rid,
                         "severidad": r["severidad"], "accion": r["accion"], "filas_evaluadas": evaluadas,
                         "filas_fallidas": len(fallidas), "estado": estado, "detalle": detalle or "; ".join(fallidas[:10])})

# 001: cada recurso automático tiene su descarga registrada y en disco
registrar("BRZ-TODAS-001", len(auto), [r.archivo for r in auto if r.url not in urls_ok])

# 002: sha256 del disco == manifest (inmutabilidad). Se recalcula para todo Bronze.
fallas_hash = []
for e in tqdm(manifest["archivos"], desc="sha256"):
    p = RAIZ / e["ruta"]
    if not p.exists() or bronze.sha256_archivo(p) != e["sha256"]:
        fallas_hash.append(e["ruta"])
registrar("BRZ-TODAS-002", len(manifest["archivos"]), fallas_hash)

# 003: ningún archivo vacío ni HTML
registrar("BRZ-TODAS-003", len(manifest["archivos"]),
          [e["ruta"] for e in manifest["archivos"] if e["bytes"] == 0 or _es_html(RAIZ / e["ruta"])])

# 004: fuentes manuales presentes
manuales = [r for r in recursos if r.modo == "manual"]
faltan = [r.archivo for r in manuales if not any(e["fuente"] == r.fuente for e in manifest["archivos"])]
registrar("BRZ-TODAS-004", len(manuales), faltan,
          "sin fuentes manuales" if not manuales else ("" if not faltan else f"pendiente copia manual: {', '.join(faltan)}"))

df_cal = pd.DataFrame(evaluaciones)
display(df_cal[["regla", "severidad", "filas_evaluadas", "filas_fallidas", "estado", "detalle"]])

sha256: 100%|██████████████████████████████████████████████████████████████████████| 1134/1134 [00:12<00:00, 90.44it/s]


,regla,severidad,filas_evaluadas,filas_fallidas,estado,detalle
0,BRZ-TODAS-001,critica,44,0,OK,
1,BRZ-TODAS-002,critica,1134,0,OK,
2,BRZ-TODAS-003,critica,1134,0,OK,
3,BRZ-TODAS-004,critica,0,0,OK,sin fuentes manuales


## 6. Bitácora de la ejecución

In [8]:
bitacora = {
    "run_id": RUN_ID, "notebook": "01_bronze_descarga", "fecha_carpeta": FECHA, "fin_utc": bronze._ahora(),
    "recursos": len(recursos), "estados": df_res["estado"].value_counts().to_dict(),
    "archivos_registrados_en_esta_ejecucion": int(df_res["archivos_registrados"].sum()),
    "bronze_total_archivos": len(manifest["archivos"]),
    "bronze_total_gb": round(sum(e["bytes"] for e in manifest["archivos"]) / 1e9, 3),
    "calidad": {ev["regla"]: ev["estado"] for ev in evaluaciones},
}
with open(ruta("bronze") / "bitacora_ejecuciones.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps(bitacora, ensure_ascii=False) + "\n")
print(json.dumps(bitacora, indent=2, ensure_ascii=False))

criticas = df_cal[(df_cal["severidad"] == "critica") & (df_cal["estado"] != "OK")]
assert criticas.empty, f"Reglas críticas fallidas:\n{criticas}"
print("\nReglas críticas: OK")

{
  "run_id": "bronze_descarga_20261121T030045Z_f2c118",
  "notebook": "01_bronze_descarga",
  "fecha_carpeta": "2026-11-20",
  "fin_utc": "2026-11-21T03:00:59+00:00",
  "recursos": 44,
  "estados": {
    "ya_existe": 44
  },
  "archivos_registrados_en_esta_ejecucion": 0,
  "bronze_total_archivos": 1134,
  "bronze_total_gb": 7.749,
  "calidad": {
    "BRZ-TODAS-001": "OK",
    "BRZ-TODAS-002": "OK",
    "BRZ-TODAS-003": "OK",
    "BRZ-TODAS-004": "OK"
  }
}

Reglas críticas: OK
